# **PINN CLÁSICA CON 6 CAPAS 64 NEURONAS**

## **1. Librerías usadas**

In [ ]:
# Qiskit (IBM) para la capa cuántica. En Colab, ejecutar una vez y reiniciar la sesión si lo pide.
!pip install -q qiskit qiskit-aer

# Opcional, para ejecutar en hardware de IBM: !pip install -q qiskit-ibm-runtime


In [ ]:
#Starndard libreries
import os
import sys
import io
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random

#import libreries of NN
import tensorflow as tf
from tensorflow import keras
import tensorflow_probability as tfp

import tensorflow as tf
tf.keras.backend.set_floatx('float64') # Fuerza todo a 64 bits globalmente

#import optimization
from scipy.optimize import minimize
from scipy.interpolate import CubicSpline
import scipy.optimize


In [ ]:
# Define la ruta
path = '/content/ABUN_PINN'

# Crea la carpeta si no existe
if not os.path.exists(path):
    os.makedirs(path)
    print(f"Carpeta '{path}' creada con éxito.")
else:
    print(f"La carpeta '{path}' ya existe.")

## **2. Entrenamiento del la PINN**

In [ ]:
# Configurar semilla
seed = 0
tf.random.set_seed(seed)
np.random.seed(seed)
random.seed(seed)

# Parámetros físicos
sigma = 1.0
k0_sim = [1.0, 2.0, 5.0]
A_sim = [1.0, 1.0, 1.0]
k_boost = 0.0
m = 1.0
c = 1.0


#Parámetros iniciales
N = 12000
L = 80 * sigma
dx = L / N
T = L/2 - 4*sigma
dt = T / N

X_min, X_max = -L/2, L/2
T_min, T_max = 0.0, T

def condition_initial(x, sigma, k0, k_boost, m, A, dx):
    k0_arr = np.array(k0).reshape(-1, 1)
    A_arr = np.array(A).reshape(-1, 1)
    x_arr = x.reshape(1, -1)

    fase_estacionaria = np.sum(A_arr * np.exp(1j * k0_arr * x_arr) + A_arr * np.exp(-1j * k0_arr * x_arr), axis=0)
    fase_estacionaria = fase_estacionaria.reshape(-1, 1)

    psi_static = np.exp(-(x**2)/(2 * sigma**2)) * fase_estacionaria
    psi_moving = psi_static * np.exp(1j * k_boost * x)
    psi_moving = psi_moving / np.sqrt(np.max(np.abs(psi_moving)**2))

    phi0 = psi_moving.real
    phi0 = phi0 / np.sqrt(np.max(np.abs(phi0)**2))

    phi_k = np.fft.fft(psi_moving.flatten())
    k_freqs = 2 * np.pi * np.fft.fftfreq(len(x), d=dx)
    omega_k = np.sqrt(k_freqs**2 + m**2)

    pi_k_calc = -1j * omega_k * phi_k
    pi0 = np.fft.ifft(pi_k_calc).real.reshape(-1, 1)

    return phi0, pi0

# Interpolación de phi y pi
x_grid = np.linspace(X_min, X_max, 4000).reshape(-1, 1)
phi0_vals, pi0_vals = condition_initial(x_grid, sigma, k0_sim, k_boost, m, A_sim, dx)
interp_phi0 = CubicSpline(x_grid.flatten(), phi0_vals.flatten(), extrapolate=True)
interp_pi0  = CubicSpline(x_grid.flatten(), pi0_vals.flatten(), extrapolate=True)

In [ ]:
class FourierEmbedding(tf.keras.layers.Layer):
    def __init__(self, num_features=128, sigma=10.0, input_dim=2, **kwargs):
        super().__init__(**kwargs)
        self.num_features = num_features
        self.sigma = sigma
        self.input_dim = input_dim

    def build(self, input_shape):
        self.B = self.add_weight(
            name='B',
            shape=(self.input_dim, self.num_features),
            initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=self.sigma),
            trainable=False,
            dtype=tf.float32
        )
        super().build(input_shape)

    def call(self, x):
        x = tf.cast(x, tf.float32)                              # ← forzar float32
        projected = 2.0 * np.pi * tf.matmul(x, self.B)
        return tf.concat([tf.cos(projected), tf.sin(projected)], axis=-1)

In [ ]:
class MLPLayer(tf.keras.layers.Layer):
    def __init__(self, units, activation='tanh', **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        glorot_init = tf.keras.initializers.GlorotNormal(seed=seed)
        self.dense = tf.keras.layers.Dense(
            self.units,
            dtype=tf.float32,
            kernel_initializer=glorot_init
        )

    def call(self, inputs):
        z = self.dense(inputs)
        return self.activation(z)

In [ ]:
@tf.keras.utils.register_keras_serializable()
class PINN(tf.keras.Model):
    def __init__(self, num_layers=5, hidden_dim=50, L=80.0, T_max=40.0, X_min=-40.0, X_max=40.0, activation='tanh',
                 use_fourier=False, num_fourier_features=64, sigma=1.0, **kwargs):
        super().__init__(**kwargs)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.L = L

        # Fourier Feature Embedding (igual que en M4, mABU y MABUR)
        #   use_fourier = True  -> se activa el embedding: (x,t) -> [cos(2π v B), sin(2π v B)]
        #   use_fourier = False -> la red recibe (x_norm, t_norm) directamente
        self.use_fourier = use_fourier
        self.num_fourier_features = num_fourier_features
        self.sigma = sigma
        self.activation = activation

        # Normalizaciones internas en Float32
        self.T_max = tf.constant(T_max, dtype=tf.float32)
        self.X_min = tf.constant(X_min, dtype=tf.float32)
        self.X_max = tf.constant(X_max, dtype=tf.float32)

    def build(self, input_shape):
        # Opcional: Embedding Fourier
        if self.use_fourier:
            self.fourier = FourierEmbedding(
                num_features=self.num_fourier_features,
                sigma=self.sigma,
                input_dim=2,
                name='fourier_embedding'
            )

        self.hidden_layers = []
        for i in range(self.num_layers):
            self.hidden_layers.append(
                MLPLayer(self.hidden_dim, activation=self.activation, name=f"MLPLayer_{i}")
            )
        self.out_layer = tf.keras.layers.Dense(1, dtype=tf.float32, kernel_initializer='glorot_uniform', name="Output")
        super().build(input_shape)

    def call(self, inputs):
        # Los inputs (x, t) llegan en float64 desde la física, los bajamos a float32 para la red
        x = tf.cast(inputs[0], tf.float32)
        t = tf.cast(inputs[1], tf.float32)

        x_norm = 2.0 * (x - self.X_min) / (self.X_max - self.X_min) - 1.0
        t_norm = t / self.T_max

        h = tf.concat([x_norm, t_norm], axis=-1)

        # Embedding o características directas
        if self.use_fourier:
            h = self.fourier(h)

        for layer in self.hidden_layers:
            h = layer(h)

        N_xt = self.out_layer(h)

        # UPCAST: Devolvemos float64 para que el Tape calcule residuos físicos con alta precisión
        return tf.cast(N_xt, tf.float64)

## **1b. VQC en Qiskit (IBM) y HQPINN (Sedykh et al., 2024)**

El circuito (*quantum depth-infused layer*: 3 qubits, d = 5, m = 2) es un `QuantumCircuit` de **Qiskit**. La capa cuántica tiene dos motores que calculan lo mismo:
- `"simulacion_exacta"`: el circuito traducido a un simulador de vector de estado en TensorFlow (derivadas de 2º orden y XLA). Es el que usa el entrenamiento, como en Sedykh et al.
- `"qiskit_aer"`: **Qiskit ejecuta el VQC** (`qiskit_aer.primitives.EstimatorV2`, gradiente por *parameter-shift*). Se usa en la celda de verificación.

In [ ]:
# =====================================================================
#  HQPINN con el VQC en QISKIT (IBM) — Sedykh et al., MLST 5, 025045 (2024)
#
#  El circuito variacional (VQC) es un QuantumCircuit de Qiskit. La capa cuántica tiene dos motores
#  que calculan exactamente lo mismo (⟨Z_q⟩ del mismo circuito, mismos pesos w):
#
#   motor = "simulacion_exacta"   -> el circuito de Qiskit traducido a un simulador de vector de estado
#                                    en TensorFlow: diferenciable a 2º orden y apto para XLA. Es el que
#                                    usa el entrenamiento (como Sedykh: simulador exacto + adjoint/backprop).
#   motor = "qiskit_aer"          -> Qiskit EJECUTA el circuito (qiskit_aer.primitives.EstimatorV2),
#                                    con gradiente por parameter-shift. Se usa en la celda de verificación.
#
#  Requisitos (Colab):  !pip install -q qiskit qiskit-aer      (opcional: qiskit-ibm-runtime)
# =====================================================================

import time
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector, Parameter
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator


# ---------- 1. Circuito de Sedykh et al. (fig. 1) en Qiskit ----------

def circuito_sedykh(n_qubits=3, d=5, m=2):
    """
    Quantum depth-infused layer:
      [Rx(w) en cada qubit + anillo de CNOT] × m
      { Rz(x) en cada qubit -> [Rx(w) en cada qubit + anillo de CNOT] × m } × d
    Devuelve (qc, x, w): x = ángulos de entrada (n·d), w = pesos variacionales (n·m·(d+1)).
    """
    n = n_qubits
    x = ParameterVector("x", n * d)
    w = ParameterVector("w", n * m * (d + 1))
    qc = QuantumCircuit(n, name="QDI")
    anillo = [(q, q + 1) for q in range(n - 1)] + ([(n - 1, 0)] if n > 2 else [])
    k = 0

    def bloque_variacional():
        nonlocal k
        for _ in range(m):
            for q in range(n):
                qc.rx(w[k], q); k += 1
            for c, t in anillo:
                qc.cx(c, t)

    bloque_variacional()
    for j in range(d):
        for q in range(n):
            qc.rz(x[j * n + q], q)
        bloque_variacional()
    return qc, x, w


def observables_z(n_qubits):
    """σz en cada qubit (convención de Qiskit: el qubit q es el carácter n−1−q de la cadena Pauli)."""
    return [SparsePauliOp("I" * (n_qubits - 1 - q) + "Z" + "I" * q) for q in range(n_qubits)]


# ---------- 2. Ejecución del VQC con Qiskit Aer (forward + parameter-shift) ----------

_PUERTAS_SHIFT = {"rx", "ry", "rz", "rxx", "ryy", "rzz", "rzx"}   # generador con autovalores ±½ -> shift ±π/2

# "auto": GPU si Qiskit Aer la ve (paquete qiskit-aer-gpu), si no CPU | "GPU" | "CPU"
DISPOSITIVO_QISKIT = "auto"

def opciones_aer(dispositivo="auto"):
    """
    Opciones de AerSimulator para ejecutar MUCHAS veces el mismo circuito con distintos parámetros:
      runtime_parameter_bind_enable=True -> Aer liga los parámetros en tiempo de ejecución (no construye
      un circuito por cada fila de valores): ~10× más rápido en CPU; en GPU además ejecuta en lotes.
    """
    from qiskit_aer import AerSimulator
    disponibles = AerSimulator().available_devices()
    if dispositivo == "auto":
        dispositivo = "GPU" if "GPU" in disponibles else "CPU"
    if dispositivo not in disponibles:
        raise ValueError(f"Qiskit Aer no ve el dispositivo {dispositivo}; disponibles: {disponibles}")
    opciones = {"method": "statevector", "device": dispositivo, "runtime_parameter_bind_enable": True}
    if dispositivo == "CPU":
        opciones["max_parallel_threads"] = 0                 # todos los núcleos
    return opciones, dispositivo


def medir_velocidad_qiskit(qc, filas=20000, semilla=0):
    """Tiempo por circuito de Qiskit Aer con las opciones de opciones_aer (para comparar CPU vs GPU)."""
    V = np.random.default_rng(semilla).uniform(-np.pi, np.pi, (filas, qc.num_parameters))
    obs = [[o] for o in observables_z(qc.num_qubits)]
    for disp in AerSimulator_dispositivos():
        est = AerEstimator(options={"backend_options": opciones_aer(disp)[0]})
        est.run([(qc, obs, V[:100])]).result()                                   # calentamiento
        t0 = time.time(); est.run([(qc, obs, V)]).result(); dt = time.time() - t0
        print(f"Qiskit Aer {disp}: {dt / filas * 1e6:.1f} µs por circuito ({filas} circuitos en {dt:.2f} s)")


def AerSimulator_dispositivos():
    from qiskit_aer import AerSimulator
    return tuple(AerSimulator().available_devices())

class EjecutorQiskit:
    """
    Ejecuta el QuantumCircuit con un EstimatorV2 de Qiskit (por defecto Aer, vector de estado exacto).
      esperanzas(X, W)                    -> ⟨Z_q⟩ para cada fila de X          [B, n]
      gradiente_parameter_shift(X, W, dy) -> (∂L/∂X [B, n_x], ∂L/∂W [n_w])
    Regla de parameter-shift (exacta para Rx/Ry/Rz):  ∂⟨Z⟩/∂θ = [⟨Z⟩(θ + π/2) − ⟨Z⟩(θ − π/2)] / 2.
    Todas las ejecuciones de un paso van en UN solo PUB (circuito, observables, matriz de valores).
    """
    def __init__(self, qc, x_params, w_params, estimador=None, max_filas=200_000):
        self.qc = qc
        self.n = qc.num_qubits
        if estimador is None:
            opciones, self.dispositivo = opciones_aer(DISPOSITIVO_QISKIT)
            estimador = AerEstimator(options={"backend_options": opciones})
            print(f"Qiskit Aer: VQC en {self.dispositivo} | opciones = {opciones}")
        else:
            self.dispositivo = "externo"
        self.estimador = estimador
        self.obs = [[o] for o in observables_z(self.n)]                        # [n, 1] -> broadcasting
        pos = {p: i for i, p in enumerate(qc.parameters)}                       # columnas en el orden de Qiskit
        faltan = [p for p in list(x_params) + list(w_params) if p not in pos]
        if faltan:
            raise ValueError(f"Parámetros que no aparecen en el circuito: {faltan[:5]}")
        self.col_x = np.array([pos[p] for p in x_params])
        self.col_w = np.array([pos[p] for p in w_params])
        self.max_filas = int(max_filas)
        self.circuitos_ejecutados = 0
        # la regla de parameter-shift exige: cada parámetro una sola vez y en una rotación Pauli
        usos = {}
        for ins in qc.data:
            for par in ins.operation.params:
                if isinstance(par, Parameter):
                    if ins.operation.name not in _PUERTAS_SHIFT:
                        raise ValueError(f"Parameter-shift no implementado para la puerta {ins.operation.name}")
                    usos[par] = usos.get(par, 0) + 1
        repetidos = [p for p, k in usos.items() if k > 1]
        if repetidos:
            raise ValueError(f"Parámetros usados más de una vez (parameter-shift no aplica): {repetidos[:5]}")

    def _valores(self, X, W):
        V = np.empty((X.shape[0], self.qc.num_parameters))
        V[:, self.col_x] = X
        V[:, self.col_w] = W
        return V

    def _ejecutar(self, V):
        """Corre el circuito para cada fila de V (en bloques de max_filas) y devuelve ⟨Z⟩ [K, n]."""
        salidas = []
        for i in range(0, V.shape[0], self.max_filas):
            bloque = V[i:i + self.max_filas]
            try:
                res = self.estimador.run([(self.qc, self.obs, bloque)]).result()
            except Exception as e:
                if self.dispositivo != "GPU":
                    raise
                print(f"Qiskit Aer en GPU falló ({type(e).__name__}: {str(e)[:120]}); se usa CPU.")
                opciones, self.dispositivo = opciones_aer("CPU")
                self.estimador = AerEstimator(options={"backend_options": opciones})
                res = self.estimador.run([(self.qc, self.obs, bloque)]).result()
            salidas.append(np.asarray(res[0].data.evs).T)
            self.circuitos_ejecutados += bloque.shape[0]
        return np.concatenate(salidas, axis=0)

    def esperanzas(self, X, W):
        X = np.asarray(X, np.float64); W = np.asarray(W, np.float64).ravel()
        return self._ejecutar(self._valores(X, W)).astype(np.float32)

    def gradiente_parameter_shift(self, X, W, dy):
        X = np.asarray(X, np.float64); W = np.asarray(W, np.float64).ravel(); dy = np.asarray(dy, np.float64)
        B, n_x = X.shape
        V0 = self._valores(X, W)                                                # [B, P]
        cols = np.concatenate([self.col_x, self.col_w])                          # parámetros a derivar
        S = len(cols)
        V = np.repeat(V0[None], 2 * S, axis=0)                                   # [2S, B, P]
        for k, col in enumerate(cols):
            V[2 * k, :, col] += np.pi / 2
            V[2 * k + 1, :, col] -= np.pi / 2
        E = self._ejecutar(V.reshape(-1, V0.shape[1])).reshape(2 * S, B, self.n)
        dZ = 0.5 * (E[0::2] - E[1::2])                                           # [S, B, n] = ∂⟨Z⟩/∂θ_k
        G = np.einsum("sbn,bn->bs", dZ, dy)                                      # regla de la cadena con dL/d⟨Z⟩
        return G[:, :n_x].astype(np.float32), G[:, n_x:].sum(axis=0).astype(np.float32)


# ---------- 3. Traductor Qiskit -> TensorFlow (simulación exacta, diferenciable) ----------

_PUERTAS_PESO = {"rx", "ry", "rz"}

def _matriz_rot(nombre, theta):
    """Matriz 2×2 (complex64) de una rotación con ángulo escalar (tensor)."""
    # tf.complex (no tf.cast) para que el gradiente respecto al ángulo real sea limpio
    cr, sr = tf.cos(theta / 2.0), tf.sin(theta / 2.0)
    c = tf.complex(cr, tf.zeros_like(cr))
    s = tf.complex(sr, tf.zeros_like(sr))
    if nombre == "rx":
        return tf.stack([tf.stack([c, -1j * s]), tf.stack([-1j * s, c])])
    if nombre == "ry":
        return tf.stack([tf.stack([c, -s]), tf.stack([s, c])])
    if nombre == "rz":
        e = tf.complex(cr, -sr)                                  # e^{-iθ/2}
        return tf.stack([tf.stack([e, tf.zeros_like(e)]), tf.stack([tf.zeros_like(e), tf.math.conj(e)])])
    raise ValueError(nombre)


class CapaCuanticaQiskit(tf.keras.layers.Layer):
    """
    Capa de Keras a partir de un QuantumCircuit de Qiskit con dos ParameterVector:
    entradas `x` (por punto) y pesos `w` (entrenables). Salida: ⟨Z_q⟩ para cada qubit.

    capa.motor = "qiskit_aer"        -> el circuito lo ejecuta Qiskit (EjecutorQiskit, parameter-shift)
    capa.motor = "simulacion_exacta" -> simulación de vector de estado en TF (traducción de abajo)

    Traducción (se hace una vez, al crear la capa):
      - tramos de puertas que solo dependen de pesos (rx/ry/rz con w, cx, cz, h, x, …) -> una matriz 2^n×2^n
        construida desde los pesos y aplicada con un matmul;
      - tramos de Rz(x) -> fase diagonal por amplitud (barato);
      - Rx/Ry(x) (otras codificaciones) -> se aplican puerta a puerta.
    Convención de qubits idéntica a Qiskit (qubit q = bit q menos significativo), así que el
    resultado coincide con qiskit.quantum_info.Statevector.
    """

    def __init__(self, qc, x_params, w_params, **kwargs):
        super().__init__(**kwargs)
        self.qc = qc
        self.n_qubits = qc.num_qubits
        self.dim = 2 ** self.n_qubits
        self.n_x, self.n_w = len(x_params), len(w_params)
        self._x_params, self._w_params = tuple(x_params), tuple(w_params)
        self.motor = "simulacion_exacta"
        self._ejecutor = None
        idx_x = {p: i for i, p in enumerate(x_params)}
        idx_w = {p: i for i, p in enumerate(w_params)}
        n = self.n_qubits
        s = np.arange(self.dim)
        self._bits = (s[:, None] >> np.arange(n)[None, :]) & 1                  # [2^n, n] (convención Qiskit)
        self.signos_z = tf.constant((1.0 - 2.0 * self._bits).astype(np.float32))

        # --- recorrer el circuito y agrupar en tramos ---
        self.tramos = []                      # ("pesos", [ops]) | ("rz_x", [(q, i_x)]) | ("rot_x", nombre, q, i_x)
        actual = None
        for ins in qc.data:
            op = ins.operation
            qs = [qc.find_bit(qb).index for qb in ins.qubits]
            nombre = op.name
            if nombre in ("barrier",):
                continue
            if nombre == "measure":
                raise ValueError("Quita las mediciones del circuito: la capa calcula ⟨Z⟩ directamente.")
            p_x, p_w = None, None
            for par in op.params:
                if hasattr(par, "parameters") and len(par.parameters) > 0:
                    if par not in idx_x and par not in idx_w:
                        raise ValueError(f"Solo se admiten parámetros simples (sin expresiones): {par}")
                    if par in idx_x: p_x = idx_x[par]
                    else: p_w = idx_w[par]
            if p_x is not None:
                if nombre == "rz":
                    if actual is None or actual[0] != "rz_x":
                        actual = ("rz_x", []); self.tramos.append(actual)
                    actual[1].append((qs[0], p_x))
                elif nombre in ("rx", "ry"):
                    self.tramos.append(("rot_x", nombre, qs[0], p_x)); actual = None
                else:
                    raise ValueError(f"Puerta de codificación no soportada: {nombre}")
            else:
                if actual is None or actual[0] != "pesos":
                    actual = ("pesos", []); self.tramos.append(actual)
                if p_w is not None:
                    if nombre not in _PUERTAS_PESO:
                        raise ValueError(f"Puerta con peso no soportada: {nombre}")
                    actual[1].append(("rot_w", nombre, qs[0], p_w))
                else:
                    # puerta fija: su matriz exacta la da Qiskit
                    from qiskit.quantum_info import Operator
                    sub = QuantumCircuit(n); sub.append(op, qs)
                    actual[1].append(("fija", Operator(sub).data.astype(np.complex64)))

        # --- optimización del grafo (menos operaciones -> compilación XLA más corta y pasos más rápidos) ---
        #  (a) puertas fijas consecutivas (p. ej. el anillo de CNOT) -> una sola matriz constante
        #  (b) R_P(θ) = cos(θ/2)·I − i·sin(θ/2)·P_q, con P_q (X, Y o Z en el qubit q) precalculada en 2^n×2^n
        paulis = {"rx": np.array([[0, 1], [1, 0]]), "ry": np.array([[0, -1j], [1j, 0]]), "rz": np.diag([1, -1])}
        self._P = {}
        for tramo in self.tramos:
            if tramo[0] != "pesos":
                continue
            fusion = []
            for o in tramo[1]:
                if o[0] == "fija" and fusion and fusion[-1][0] == "fija":
                    fusion[-1] = ("fija", (o[1] @ fusion[-1][1]).astype(np.complex64))
                else:
                    fusion.append(o)
                if o[0] == "rot_w" and (o[1], o[2]) not in self._P:
                    M = np.array([[1.0]])
                    for k in reversed(range(n)):                                # kron(Q_{n-1}, …, Q_0)
                        M = np.kron(M, paulis[o[1]] if k == o[2] else np.eye(2))
                    self._P[(o[1], o[2])] = (tf.constant(M.real.astype(np.float32)), tf.constant(M.imag.astype(np.float32)))
            tramo[1][:] = [("fija", (tf.constant(o[1].real.astype(np.float32)), tf.constant(o[1].imag.astype(np.float32))))
                           if o[0] == "fija" else o for o in fusion]
        self._I = tf.eye(self.dim, dtype=tf.float32)

    def build(self, input_shape):
        init = tf.keras.initializers.RandomUniform(minval=-np.pi, maxval=np.pi)
        self.w = self.add_weight(name="w_qiskit", shape=(self.n_w,), initializer=init, trainable=True, dtype=tf.float32)
        super().build(input_shape)

    # --- utilidades sobre matrices 2^n×2^n (convención Qiskit: qubit 0 = factor más a la derecha) ---
    def _expandir(self, U2, q):
        n = self.n_qubits
        M = None
        for k in reversed(range(n)):                  # kron(Q_{n-1}, …, Q_0)
            f = U2 if k == q else tf.eye(2, dtype=tf.complex64)
            M = f if M is None else tf.reshape(tf.einsum('ij,kl->ikjl', M, f), [M.shape[0] * 2, M.shape[1] * 2])
        return M

    def _unitaria_tramo(self, ops):
        """Unitaria del tramo como par real (A, B) con U = A + iB (sin aritmética compleja: grafo más liviano)."""
        U = None
        for o in ops:
            if o[0] == "fija":
                G = o[1]
            else:
                _, nombre, q, i_w = o
                th = self.w[i_w] / 2.0
                c, s = tf.cos(th), tf.sin(th)
                Pr, Pi = self._P[(nombre, q)]                       # R_P(θ) = c·I − i·s·P,  P = Pr + i·Pi
                G = (c * self._I + s * Pi, -s * Pr)
            if U is None:
                U = G
            else:                                                   # G·U con pares reales
                U = (tf.matmul(G[0], U[0]) - tf.matmul(G[1], U[1]), tf.matmul(G[0], U[1]) + tf.matmul(G[1], U[0]))
        return U

    @staticmethod
    def _aplicar(re, im, U):
        A, Bm = U
        return (tf.matmul(re, A, transpose_b=True) - tf.matmul(im, Bm, transpose_b=True),
                tf.matmul(re, Bm, transpose_b=True) + tf.matmul(im, A, transpose_b=True))

    def _rot_por_punto(self, re, im, nombre, q, ang):
        """Rx/Ry(x) con ángulo por punto sobre el qubit q (convención Qiskit)."""
        n = self.n_qubits
        shape = [-1, 2 ** (n - 1 - q), 2, 2 ** q]
        r0, r1 = tf.unstack(tf.reshape(re, shape), axis=2)
        i0, i1 = tf.unstack(tf.reshape(im, shape), axis=2)
        c = tf.reshape(tf.cos(ang / 2.0), [-1, 1, 1]); s = tf.reshape(tf.sin(ang / 2.0), [-1, 1, 1])
        if nombre == "rx":
            nr0, ni0 = c * r0 + s * i1, c * i0 - s * r1
            nr1, ni1 = c * r1 + s * i0, c * i1 - s * r0
        else:  # ry
            nr0, ni0 = c * r0 - s * r1, c * i0 - s * i1
            nr1, ni1 = s * r0 + c * r1, s * i0 + c * i1
        unir = lambda a, b: tf.reshape(tf.stack([a, b], axis=2), [-1, self.dim])
        return unir(nr0, nr1), unir(ni0, ni1)

    def ejecutor(self):
        if self._ejecutor is None:
            self._ejecutor = EjecutorQiskit(self.qc, self._x_params, self._w_params)
        return self._ejecutor

    def call(self, angulos):
        if self.motor == "qiskit_aer":
            return self._call_qiskit(angulos)
        if self.motor != "simulacion_exacta":
            raise ValueError('motor debe ser "qiskit_aer" o "simulacion_exacta"')
        return self._call_simulacion(angulos)

    def _call_qiskit(self, angulos):
        """⟨Z⟩ ejecutando el QuantumCircuit con Qiskit Aer; gradiente por parameter-shift (también Qiskit)."""
        ej, n = self.ejecutor(), self.n_qubits

        @tf.custom_gradient
        def vqc_qiskit(ang, w):
            y = tf.numpy_function(ej.esperanzas, [ang, w], tf.float32)
            y = tf.ensure_shape(y, [None, n])

            def grad(dy):
                gx, gw = tf.numpy_function(ej.gradiente_parameter_shift, [ang, w, dy], [tf.float32, tf.float32])
                return tf.reshape(gx, tf.shape(ang)), tf.reshape(gw, tf.shape(w))
            return y, grad

        return vqc_qiskit(tf.cast(angulos, tf.float32), tf.identity(self.w))

    def _call_simulacion(self, angulos):
        angulos = tf.cast(angulos, tf.float32)
        B = tf.shape(angulos)[0]
        re = tf.one_hot(tf.zeros([B], tf.int32), self.dim, dtype=tf.float32)    # |0…0⟩
        im = tf.zeros_like(re)
        z = tf.constant((1.0 - 2.0 * self._bits).astype(np.float32))           # [2^n, n]
        for tramo in self.tramos:
            if tramo[0] == "pesos":
                re, im = self._aplicar(re, im, self._unitaria_tramo(tramo[1]))
            elif tramo[0] == "rz_x":
                # Rz(φ) = diag(e^{−iφ/2}, e^{+iφ/2}) -> fase θ_s = −½ Σ φ_q z_q(s)
                qs = [q for q, _ in tramo[1]]; ix = [i for _, i in tramo[1]]
                phi = tf.gather(angulos, ix, axis=1)                             # [B, k]
                theta = -0.5 * tf.matmul(phi, tf.transpose(tf.gather(z, qs, axis=1)))
                c, s = tf.cos(theta), tf.sin(theta)
                re, im = re * c - im * s, re * s + im * c
            else:
                _, nombre, q, i_x = tramo
                re, im = self._rot_por_punto(re, im, nombre, q, angulos[:, i_x])
        prob = tf.square(re) + tf.square(im)
        return tf.matmul(prob, self.signos_z)                                   # [B, n]: ⟨Z_0⟩ … ⟨Z_{n−1}⟩ (orden de Qiskit)


# ---------- 4. Evaluación con primitivas de Qiskit (verificación / ejecución) ----------

def evaluar_con_qiskit(qc, x_params, w_params, x_vals, w_vals, shots=None, estimador=None):
    """
    ⟨Z_q⟩ del circuito para cada fila de x_vals (pesos w_vals fijos), con un Estimator de Qiskit.
      estimador=None, shots=None -> StatevectorEstimator exacto
      estimador=None, shots=N    -> StatevectorEstimator con precisión 1/sqrt(N) (ruido de muestreo de N shots)
      estimador=...              -> cualquier EstimatorV2 (p. ej. Aer, o qiskit_ibm_runtime con un backend de IBM;
                                    en hardware el circuito debe transpilarse antes al backend: ISA)
    Devuelve un arreglo [B, n_qubits] con ⟨Z_0⟩ … ⟨Z_{n−1}⟩ (orden de Qiskit).
    """
    if estimador is None:
        estimador = StatevectorEstimator()
    x_vals = np.atleast_2d(np.asarray(x_vals, dtype=float))
    w_vals = np.asarray(w_vals, dtype=float).ravel()
    B = x_vals.shape[0]
    en_circuito = set(qc.parameters)                                            # ignora parámetros que no aparecen
    valores = {p: x_vals[:, i] for i, p in enumerate(x_params) if p in en_circuito}
    valores.update({p: np.full(B, w_vals[i]) for i, p in enumerate(w_params) if p in en_circuito})
    obs = [[o] for o in observables_z(qc.num_qubits)]                           # forma [n, 1] -> broadcasting [n, B]
    precision = None if shots is None else 1.0 / np.sqrt(shots)
    trabajo = estimador.run([(qc, obs, valores)], precision=precision) if precision else estimador.run([(qc, obs, valores)])
    return np.asarray(trabajo.result()[0].data.evs).T                            # [B, n]


def verificar_capa_contra_qiskit(capa, n_muestras=4, semilla=0):
    """Compara la capa de TensorFlow con Statevector de Qiskit para ángulos aleatorios."""
    rng = np.random.default_rng(semilla)
    x_vals = rng.uniform(-np.pi, np.pi, size=(n_muestras, capa.n_x)).astype(np.float32)
    salida_tf = capa._call_simulacion(tf.constant(x_vals)).numpy()
    salida_qk = evaluar_con_qiskit(capa.qc, capa._x_params, capa._w_params, x_vals, capa.w.numpy())
    err = float(np.max(np.abs(salida_tf - salida_qk)))
    print(f"Capa TF vs Qiskit (Statevector): error máximo = {err:.2e}")
    return err


def verificar_gradiente_qiskit(capa, n_muestras=8, semilla=0):
    """Gradiente por parameter-shift (Qiskit Aer) vs autodiferenciación de la simulación exacta."""
    rng = np.random.default_rng(semilla)
    ang = tf.constant(rng.uniform(-np.pi, np.pi, size=(n_muestras, capa.n_x)).astype(np.float32))
    dy = tf.constant(rng.normal(size=(n_muestras, capa.n_qubits)).astype(np.float32))
    res = {}
    for motor in ("qiskit_aer", "simulacion_exacta"):
        capa.motor = motor
        with tf.GradientTape() as tape:
            tape.watch(ang)
            y = capa(ang)
            L = tf.reduce_sum(y * dy)
        res[motor] = tape.gradient(L, [ang, capa.w])
    capa.motor = "simulacion_exacta"
    ex = float(np.max(np.abs(res["qiskit_aer"][0] - res["simulacion_exacta"][0])))
    ew = float(np.max(np.abs(res["qiskit_aer"][1] - res["simulacion_exacta"][1])))
    print(f"Gradiente parameter-shift (Qiskit) vs autodiff (TF): error máx. entradas = {ex:.2e} | pesos = {ew:.2e}")
    return ex, ew


# ---------- 5. HQPINN con la capa de Qiskit ----------

@tf.keras.utils.register_keras_serializable()
class HQPINN(tf.keras.Model):
    """
    Copia de PINN (misma MLP, normalización, embedding opcional y salida en float64) con la
    "red híbrida en paralelo" de Sedykh et al. 2024 al final; el circuito se define en Qiskit.
    El kernel de la capa Output sigue siendo trainable_variables[-2] (lo usa el LRA de train_step).
    """
    def __init__(self, num_layers=5, hidden_dim=50, L=80.0, T_max=40.0, X_min=-40.0, X_max=40.0, activation='tanh',
                 use_fourier=False, num_fourier_features=64, sigma=1.0, n_qubits=3, d=5, m=2, **kwargs):
        super().__init__(**kwargs)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.L = L
        self.use_fourier = use_fourier
        self.num_fourier_features = num_fourier_features
        self.sigma = sigma
        self.activation = activation
        self.n_qubits, self.d, self.m = n_qubits, d, m

        self.T_max = tf.constant(T_max, dtype=tf.float32)
        self.X_min = tf.constant(X_min, dtype=tf.float32)
        self.X_max = tf.constant(X_max, dtype=tf.float32)

    def build(self, input_shape):
        if self.use_fourier:
            self.fourier = FourierEmbedding(
                num_features=self.num_fourier_features, sigma=self.sigma, input_dim=2, name='fourier_embedding'
            )
        self.hidden_layers = []
        for i in range(self.num_layers):
            self.hidden_layers.append(MLPLayer(self.hidden_dim, activation=self.activation, name=f"MLPLayer_{i}"))

        # Circuito de Qiskit (Sedykh) -> capa de Keras
        qc, x_par, w_par = circuito_sedykh(self.n_qubits, self.d, self.m)
        self.pre_cuantica = tf.keras.layers.Dense(len(x_par) + 1, dtype=tf.float32,
                                                  kernel_initializer='glorot_uniform', name="Pre_cuantica")
        self.vqc = CapaCuanticaQiskit(qc, x_par, w_par, name="VQC_Qiskit", dtype=tf.float32)   # float32: sin conversiones a float64
        # La capa Output se crea al final para que su kernel sea trainable_variables[-2] (LRA)
        self.out_layer = tf.keras.layers.Dense(1, dtype=tf.float32, kernel_initializer='glorot_uniform', name="Output")
        super().build(input_shape)

    def usar_motor(self, motor):
        """"simulacion_exacta" (TF, rápido, derivadas de 2º orden) | "qiskit_aer" (Qiskit ejecuta el VQC)."""
        self.vqc.motor = motor

    def parte_clasica(self, inputs):
        """(x, t) -> salida de Pre_cuantica: [ángulos del circuito | unidad clásica]."""
        x = tf.cast(inputs[0], tf.float32)
        t = tf.cast(inputs[1], tf.float32)

        x_norm = 2.0 * (x - self.X_min) / (self.X_max - self.X_min) - 1.0
        t_norm = t / self.T_max
        h = tf.concat([x_norm, t_norm], axis=-1)
        if self.use_fourier:
            h = self.fourier(h)
        for layer in self.hidden_layers:
            h = layer(h)
        return self.pre_cuantica(h)

    def call(self, inputs):
        z = self.parte_clasica(inputs)
        n_ang = self.vqc.n_x
        z_cuantico = self.vqc(z[:, :n_ang])                  # ⟨Z_q⟩ de cada qubit
        z_clasico = z[:, n_ang:]                             # unidad clásica en paralelo
        N_xt = self.out_layer(tf.concat([z_cuantico, z_clasico], axis=-1))
        return tf.cast(N_xt, tf.float64)


def predecir_con_qiskit(model, x, t, shots=None, estimador=None):
    """
    φ(x, t) de una HQPINN entrenada evaluando la capa cuántica con Qiskit (no con la simulación de TF):
    parte clásica en TF -> ⟨Z⟩ con un Estimator de Qiskit -> capa Output en TF.
    shots=None: exacto | shots=N: ruido de muestreo de N shots | estimador: Aer o un backend de IBM.
    """
    z = model.parte_clasica([x, t])
    n_ang = model.vqc.n_x
    ang = z[:, :n_ang].numpy()
    q = evaluar_con_qiskit(model.vqc.qc, model.vqc._x_params, model.vqc._w_params, ang,
                           model.vqc.w.numpy(), shots=shots, estimador=estimador)
    out = model.out_layer(tf.concat([tf.constant(q, tf.float32), z[:, n_ang:]], axis=-1))
    return tf.cast(out, tf.float64).numpy()


In [ ]:
def pde_loss_causal(model, x_f, t_f, c_val, m_val, epsilon=100.0, num_chunks=10):
    t_flat = tf.reshape(t_f, [-1])
    sorted_indices = tf.argsort(t_flat)

    t_f_sorted = tf.gather(t_f, sorted_indices)
    x_f_sorted = tf.gather(x_f, sorted_indices)

    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch([x_f_sorted, t_f_sorted])
        with tf.GradientTape(persistent=True) as tape1:
            tape1.watch([x_f_sorted, t_f_sorted])
            phi = model([x_f_sorted, t_f_sorted])

        phi_x = tape1.gradient(phi, x_f_sorted)
        phi_t = tape1.gradient(phi, t_f_sorted)

    phi_xx = tape2.gradient(phi_x, x_f_sorted)
    phi_tt = tape2.gradient(phi_t, t_f_sorted)
    del tape1, tape2

    # 3. RESIDUO BRUTO (Element-wise)
    c_tf = tf.cast(c_val, tf.float64)
    m_tf = tf.cast(m_val, tf.float64)

    # Residuo punto por punto (vector columna)
    residual = phi_tt - tf.square(c_tf) * phi_xx + tf.square(m_tf) * phi
    squared_residual = tf.square(residual)

    N_total = tf.shape(squared_residual)[0]
    chunk_size = N_total // num_chunks

    N_exact = chunk_size * num_chunks
    sq_res_trunc = squared_residual[:N_exact]
    chunks = tf.reshape(sq_res_trunc, [num_chunks, chunk_size])

    L_i = tf.reduce_mean(chunks, axis=1)
    cumulative_loss = tf.math.cumsum(L_i, exclusive=True)

    epsilon_tf = tf.constant(epsilon, dtype=tf.float64)
    causal_weights = tf.exp(-epsilon_tf * cumulative_loss)

    causal_weights = tf.stop_gradient(causal_weights)
    causal_loss = tf.reduce_mean(causal_weights * L_i)

    return causal_loss

def initial_condition_loss(model, x_ic, t_ic, phi0_tf, pi0_tf):
    with tf.GradientTape() as tape:
        tape.watch(t_ic)
        phi_pred = model([x_ic, t_ic])
    phi_t_pred = tape.gradient(phi_pred, t_ic)

    loss_phi = tf.reduce_mean(tf.square(phi_pred - phi0_tf))
    loss_pi  = tf.reduce_mean(tf.square(phi_t_pred - pi0_tf))
    return loss_phi, loss_pi

def boundary_condition_loss(model, t_bc, x_bc_left, x_bc_right):
    with tf.GradientTape() as tape_l:
        tape_l.watch(x_bc_left)
        phi_left = model([x_bc_left, t_bc])
    phi_x_left = tape_l.gradient(phi_left, x_bc_left)

    with tf.GradientTape() as tape_r:
        tape_r.watch(x_bc_right)
        phi_right = model([x_bc_right, t_bc])
    phi_x_right = tape_r.gradient(phi_right, x_bc_right)

    loss_val = tf.reduce_mean(tf.square(phi_left - phi_right))
    loss_der = tf.reduce_mean(tf.square(phi_x_left - phi_x_right))

    return loss_val + loss_der

In [ ]:
@tf.function(jit_compile=True ,reduce_retracing=True)
def train_step(model, optimizer, x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc, c_tf, m_tf, alpha_lra,
               c_phi=0.0, c_pi=0.0, c_bc = 0.0, w_tensor=[1.0 ,1.0 ,1.0 ,1.0], epsilon = 1.0, num_chunks = 16):
    with tf.GradientTape(persistent=True) as tape:
        lf = pde_loss_causal(model, x_f, t_f, c_tf, m_tf, epsilon= epsilon, num_chunks= num_chunks)
        l_phi, l_pi = initial_condition_loss(model, x_ic, t_ic, phi0, pi0)
        lb = boundary_condition_loss(model, t_bc, x_bc_l, x_bc_r)

        total_loss = w_tensor[0]*lf + w_tensor[1]*l_phi + w_tensor[2]*l_pi + w_tensor[3]*lb
        tf.debugging.check_numerics(total_loss, "total_loss NaN/Inf")

    #Algoritmo de LRA
    target_layer_vars = model.trainable_variables[-2]

    grad_f = tape.gradient(lf, target_layer_vars)
    grad_phi = tape.gradient(l_phi, target_layer_vars)
    grad_pi = tape.gradient(l_pi, target_layer_vars)
    grad_b = tape.gradient(lb, target_layer_vars)

    max_f = tf.cast(tf.reduce_max(tf.abs(grad_f)), tf.float64)
    mean_phi = tf.cast(tf.reduce_mean(tf.abs(grad_phi)), tf.float64)
    mean_pi = tf.cast(tf.reduce_mean(tf.abs(grad_pi)), tf.float64)
    mean_b = tf.cast(tf.reduce_mean(tf.abs(grad_b)), tf.float64)

    eps = tf.constant(np.finfo(float).eps, dtype=tf.float64)

    #Por si las moscas
    den_phi = mean_phi + c_phi*max_f
    den_pi = mean_pi + c_pi*max_f
    den_bc = mean_b + c_bc-max_f

    # Calculamos los multiplicadores crudos
    lamb_phi = max_f / (den_phi + eps)
    lamb_pi = max_f / (den_pi + eps)
    lamb_b = max_f / (den_bc + eps)

    lamb_phi = tf.clip_by_value(lamb_phi, 1e-1, 1e+2)
    lamb_pi  = tf.clip_by_value(lamb_pi,  1e-2, 1e+1)
    lamb_b  = tf.clip_by_value(lamb_b,  1e-3, 1e-1)

    # Actualizamos los pesos mediante Media Móvil (EMA) directamente en la GPU
    w_phi_new = (1.0 - alpha_lra) * w_tensor[1] + alpha_lra * lamb_phi
    w_pi_new = (1.0 - alpha_lra) * w_tensor[2] + alpha_lra * lamb_pi
    w_bc_new = (1.0 - alpha_lra) * w_tensor[3] + alpha_lra * lamb_b

    # Asignamos los nuevos pesos a la variable global (Entrarán en efecto en la próxima época)
    w_tensor.assign([w_tensor[0], w_phi_new, w_pi_new, w_bc_new])

    # OPTIMIZACIÓN
    grads = tape.gradient(total_loss, model.trainable_variables)
    del tape

    grads = [tf.zeros_like(v) if g is None else g for g, v in zip(grads, model.trainable_variables)]
    grads, gnorm = tf.clip_by_global_norm(grads, 50.0)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    return total_loss, lf, l_phi, l_pi, lb, gnorm

def resample_all_focused(n_phys, n_bc, n_ic, focus=True):
    """ Muestreo focalizado: cono+global para PDE, gaussiano+uniforme para IC """

    # ============ PDE: 80% cono de luz + 20% global ============
    if focus:
        n_cone = int(0.8 * n_phys)
        n_global = n_phys - n_cone

        # Cono (alta densidad donde hay acción)
        t_cone = np.random.uniform(0.0, T_max, (n_cone, 1)).astype(np.float64)
        margen = 5.0
        radios = sigma + c * t_cone + margen
        radios = np.clip(radios, 0.0, X_max)
        u = np.random.uniform(-1.0, 1.0, (n_cone, 1)).astype(np.float64)
        x_cone = u * radios

        # Global (enseña φ≈0 fuera del cono)
        t_global = np.random.uniform(0.0, T_max, (n_global, 1)).astype(np.float64)
        x_global = np.random.uniform(X_min, X_max, (n_global, 1)).astype(np.float64)

        x_f_new = np.concatenate([x_cone, x_global], axis=0)
        t_f_new = np.concatenate([t_cone, t_global], axis=0)
    else:
        t_f_new = np.random.uniform(0.0, T_max, (n_phys, 1)).astype(np.float64)
        x_f_new = np.random.uniform(X_min, X_max, (n_phys, 1)).astype(np.float64)

    # ============ BC: fronteras ============
    t_bc_new = np.random.uniform(0.0, T_max, (n_bc, 1)).astype(np.float64)
    x_bc_l_new = np.full((n_bc, 1), X_min, dtype=np.float64)
    x_bc_r_new = np.full((n_bc, 1), X_max, dtype=np.float64)

    # ============ IC: 80% gaussiano + 20% uniforme ============
    if focus:
        n_ic_focus = int(0.8 * n_ic)
        n_ic_unif  = n_ic - n_ic_focus
        x_ic_focus = np.clip(np.random.normal(0.0, 4.0 * sigma, (n_ic_focus, 1)),
                             X_min, X_max)
        x_ic_unif  = np.random.uniform(X_min, X_max, (n_ic_unif, 1))
        x_ic_new   = np.concatenate([x_ic_focus, x_ic_unif], axis=0).astype(np.float64)
    else:
        x_ic_new = np.random.uniform(X_min, X_max, (n_ic, 1)).astype(np.float64)

    t_ic_new = np.zeros((n_ic, 1), dtype=np.float64)
    phi0_new = interp_phi0(x_ic_new.flatten()).reshape(-1, 1).astype(np.float64)
    pi0_new  = interp_pi0(x_ic_new.flatten()).reshape(-1, 1).astype(np.float64)

    return (tf.constant(x_f_new), tf.constant(t_f_new),
            tf.constant(x_ic_new), tf.constant(t_ic_new),
            tf.constant(phi0_new), tf.constant(pi0_new),
            tf.constant(x_bc_l_new), tf.constant(x_bc_r_new), tf.constant(t_bc_new))

In [ ]:
@tf.function(reduce_retracing=True)
def get_loss_and_grads_lbfgs(x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc, c_tf, m_tf, epsilon, num_chunks):
    with tf.GradientTape() as tape:
        lf = pde_loss_causal(model, x_f, t_f, c_tf, m_tf, epsilon = epsilon, num_chunks= num_chunks)
        l_phi, l_pi = initial_condition_loss(model, x_ic, t_ic, phi0, pi0)
        lb = boundary_condition_loss(model, t_bc, x_bc_l, x_bc_r)
        total_loss = lf + l_phi + l_pi + lb

    grads = tape.gradient(total_loss, model.trainable_variables)

    # Aplanamos los gradientes y los escalamos a Float64 para L-BFGS (Scipy lo requiere)
    flat_grads = tf.cast(tf.concat([tf.reshape(g, [-1]) for g in grads if g is not None], axis=0), tf.float64)

    return total_loss, lf, l_phi, l_pi, lb, flat_grads


def set_flat_weights(model, flat_weights):
    flat_weights = tf.cast(flat_weights, tf.float32)
    start = 0
    for w in model.trainable_variables:
        shape = w.shape
        size = tf.reduce_prod(shape)
        new_w = tf.reshape(flat_weights[start:start + size], shape)
        w.assign(new_w)
        start += size


def get_flat_weights(model):
    weights = [tf.reshape(tf.cast(w, tf.float64), [-1]) for w in model.trainable_variables]
    return tf.concat(weights, axis=0).numpy()


def lbfgs_objective(flat_weights, x_f, t_f, x_ic, t_ic, phi0, pi0,
                    x_bc_l, x_bc_r, t_bc, c_tf, m_tf, epsilon, num_chunks):
    set_flat_weights(model, flat_weights)
    t_loss, lf, l_phi, l_pi, lb, grads = get_loss_and_grads_lbfgs(
        x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc,
        c_tf, m_tf, epsilon, num_chunks
    )

    loss_val = t_loss.numpy().astype(np.float64)
    grads_val = grads.numpy().astype(np.float64)

    global lbfgs_iter
    lbfgs_iter += 1

    if lbfgs_iter % 100 == 0:
        print(f"L-BFGS Iter {lbfgs_iter} | Total Loss={loss_val:.2e} | "
              f"PDE={lf:.2e} | PHI={l_phi:.2e} | PI={l_pi:.2e} | BC={lb:.2e}")
        loss_history["epoch"].append(epochs_adam + lbfgs_iter)
        loss_history["total"].append(loss_val)
        loss_history["pde"].append(lf.numpy())
        loss_history["phi"].append(l_phi.numpy())
        loss_history["pi"].append(l_pi.numpy())
        loss_history["bc"].append(lb.numpy())

    return loss_val, grads_val

In [ ]:
#Seed
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Parámetros de ejecución
neurons = 64
num_blocks = 6
epochs_adam = 100000
max_iter_lbfgs = 0
print_every = 1000
as_freq = 1
stop_resampling = epochs_adam
n_phys = N
n_bc = N // 4
n_ic = N // 2

#LRA
alpha_lra = 0.9
w_tensor = tf.Variable([1.0, 1.0, 1.0, 1e-1], dtype=tf.float64)

#SIREN
omega_0 = 15.0

#Causal Training
epsilon = 10.0
num_chunks = 16

# Modelo: "PINN" (clásica) | "HQPINN" (híbrida cuántica, Sedykh et al. 2024)
MODELO = "HQPINN"
n_qubits = 3        # Sedykh et al.: 3 qubits
d_reupload = 5      # Sedykh et al.: d = 5 repeticiones de codificación (n_qubits·d ángulos de entrada)
m_var = 2           # Sedykh et al.: m = 2 repeticiones de la capa variacional

# Instanciar modelo
if MODELO == "PINN":
    model = PINN(
        num_layers=num_blocks,
        hidden_dim=neurons,
        use_fourier=True,          # True = activa el Embedding | False = lo desactiva
        num_fourier_features=256,
        sigma=1.0
    )
elif MODELO == "HQPINN":
    model = HQPINN(
        num_layers=num_blocks,
        hidden_dim=neurons,
        use_fourier=True,          # True = activa el Embedding | False = lo desactiva
        num_fourier_features=256,
        sigma=1.0,
        n_qubits=n_qubits, d=d_reupload, m=m_var
    )
else:
    raise ValueError('MODELO debe ser "PINN" o "HQPINN"')

# Forzar inicialización construyendo las capas
x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = resample_all_focused(n_phys, n_bc, n_ic, focus=True)
_ = model([x_f, t_f])

# Optimizador
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-3, decay_steps=2000, decay_rate=0.9, staircase=False
)
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
optimizer.build(model.trainable_variables)

# Historial principal con "gnorm" inicializado correctamente
loss_history = {"epoch": [], "total": [], "pde": [], "phi": [], "pi": [] ,"bc": [], "lambda_phi": [], "lambda_pi":[], "lambda_bc": [], "gnorm": [], "time": []}

# Funciones de pérdida

c_tf = tf.cast(c, tf.float64)
m_tf = tf.cast(m, tf.float64)

# Muestreo inicial
x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = resample_all_focused(n_phys, n_bc, n_ic, focus=True)

start_time = time.time()

print(" FASE 1: Exploración Global con Adam + LRA")

stop_focus = epochs_adam * 0.8

for epoch in range(1, epochs_adam + 1):

    if epoch % as_freq == 0 and epoch <= stop_resampling:
        # Usar muestreo enfocado al principio, muestreo global al final
        current_focus = True if epoch <= stop_focus else False

        x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = resample_all_focused(
            n_phys, n_bc, n_ic, focus=current_focus
        )

    t_loss, lf, l_phi, l_pi, lb, gnorm = train_step(
        model, optimizer, x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc, c_tf, m_tf, w_tensor = w_tensor, alpha_lra = alpha_lra, epsilon = epsilon, num_chunks = num_chunks
    )

    if epoch % print_every == 0 or epoch == 1:
        elapsed = time.time() - start_time
        print(f"AdamW Ep {epoch}/{epochs_adam} | Total={t_loss:.2e} | PDE={lf:.2e} | PHI={l_phi:.2e} | PI = {l_pi:.2e} | BC={lb:.2e} | w_phi={w_tensor[1]:.2e} | w_pi={w_tensor[2]:.2e} | w_bc={w_tensor[3]:.2e} | gnorm ={gnorm:.2e} | LR={lr_schedule(optimizer.iterations):.2e} | Time: {elapsed:.1f}s")

        # Guardado del histórico para AdamW
        loss_history["epoch"].append(epoch)
        loss_history["total"].append(t_loss.numpy())
        loss_history["pde"].append(lf.numpy())
        loss_history["phi"].append((l_phi).numpy())
        loss_history["pi"].append((l_pi).numpy())
        loss_history["bc"].append(lb.numpy())
        loss_history["lambda_phi"].append(w_tensor[1].numpy())
        loss_history["lambda_pi"].append(w_tensor[2].numpy())
        loss_history["lambda_bc"].append(w_tensor[3].numpy())
        loss_history["gnorm"].append(gnorm.numpy())
        loss_history["time"].append(round(elapsed, 1))


if max_iter_lbfgs == 0:
  pass
else:
  print("\n FASE 2: Refinamiento Fino con L-BFGS")
  lbfgs_iter = 0
  initial_weights = get_flat_weights(model)


  results = scipy.optimize.minimize(
      fun=lbfgs_objective,
      x0=initial_weights,
      args=(x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc, c_tf, m_tf, epsilon, num_chunks),
      method='L-BFGS-B',
      jac=True,
      options={
          'maxiter': max_iter_lbfgs,
          'maxfun': max_iter_lbfgs,
          'ftol': 1.0 * np.finfo(float).eps,
          'gtol': 1e-8
      }
  )
  set_flat_weights(model, tf.convert_to_tensor(results.x, dtype=tf.float64))
  print(f"\nL-BFGS Finalizado: {results.message}")
  print(f"Pérdida final alcanzada: {results.fun:.2e}")

# GUARDADO
valid_history = {k: v for k, v in loss_history.items() if len(v) > 0}
min_len = min([len(v) for v in valid_history.values()])
clean_history = {k: v[:min_len] for k, v in valid_history.items()}
df_loss = pd.DataFrame(clean_history)
df_loss.to_csv("training_loss_history_AbuPINN.csv", index=False)

# Guardamos los pesos del modelo
model.save_weights("AbuPINN_ultimate_weights.weights.h5")

print(f"Entrenamiento finalizado exitosamente y guardado (Filas guardadas: {min_len}).")

In [ ]:
import matplotlib.pyplot as plt

# Gráfica de Convergencia
plt.figure(figsize=(10, 5))

# Paleta de tonos pastel enfocada en aqua, verdes y azules
color_total  = "#000000" # Negro intenso
color_pde    = "#E63946" # Rojo carmesí vibrante
color_ic_phi = "#2A9D8F" # Verde esmeralda sólido
color_ic_pi  = "#F4A261" # Naranja fuerte
color_bc     = "#4361EE" # Azul rey puro

plt.semilogy(loss_history["epoch"], loss_history["total"], color=color_total, label="Total Loss", linewidth=1)
plt.semilogy(loss_history["epoch"], loss_history["pde"], color=color_pde, label="PDE Loss (Física)", linestyle="--", alpha=0.9, linewidth=1)
plt.semilogy(loss_history["epoch"], loss_history["phi"], color=color_ic_phi, label="IC Loss phi (Inicial)", linestyle="--", alpha=0.9, linewidth=1)
plt.semilogy(loss_history["epoch"], loss_history["phi"], color=color_ic_pi, label="IC Loss pi (Inicial)", linestyle="--", alpha=0.9, linewidth=1)
plt.semilogy(loss_history["epoch"], loss_history["bc"], color=color_bc, label="BC Loss (Frontera)", linestyle="--", alpha=0.9, linewidth=1)

plt.title("Historial de Entrenamiento PINN", fontsize=14, pad=15)
plt.xlabel("Épocas", fontsize=11)
plt.ylabel("Pérdida", fontsize=11)

# Estilizar la leyenda para que se vea más limpia
plt.legend(frameon=True, facecolor='#ffffff', edgecolor='none', fontsize=10)

# Suavizar la cuadrícula y darle un fondo sutil a los ejes para resaltar los pasteles
plt.grid(True, which="both", ls="--", alpha=0.5, color='#E0E0E0')
plt.gca().set_facecolor('#FAFAFA')

# Quitar los bordes superior y derecho para un look más moderno
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['left'].set_color('#DDDDDD')
plt.gca().spines['bottom'].set_color('#DDDDDD')

plt.show()

In [ ]:
# Nombre correcto basado en tu captura de archivos: AbuPINN (sin 'n' extra)
nombre_archivo = f"PINN_ultimate_weights.weights.h5"
ruta_completa = os.path.join("/content", nombre_archivo)

if os.path.exists(ruta_completa):
    try:
        model.load_weights(ruta_completa)
        print(f"   -> ¡ÉXITO! Pesos cargados desde: {nombre_archivo}")
    except Exception as e:
        print(f"\n[ERROR CRÍTICO]: El archivo existe pero no pudo ser leído: {e}")
else:
    print(f"\n[ERROR]: El archivo '{nombre_archivo}' NO existe en /content/ABUN_PINN/")
    print("Archivos encontrados en la carpeta:")
    print(os.listdir("/content/ABUN_PINN/"))
    # No uses sys.exit() aquí si quieres continuar con pesos aleatorios
    print("   -> Procediendo con inicialización aleatoria.")

In [ ]:
# Parámetros físicos
sigma = 1.0
k0_sim = [1.0, 2.0, 5.0]
A_sim = [1.0, 1.0, 1.0]
k_boost = 0.0
m = 1.0
c = 1.0

# Parámetros numéricos
N = 4096                    # puntos espaciales
L = 80 * sigma              # tamaño del dominio
T_max = L/2 - 4*sigma       # tiempo final (evita que el frente toque el borde)
n_t = 200                   # snapshots a guardar

# ============== MALLADO ==============
x = np.linspace(-L/2, L/2, N, endpoint=False)
dx = x[1] - x[0]
k = 2 * np.pi * np.fft.fftfreq(N, d=dx)

# Frecuencia angular de Klein-Gordon: cada modo k oscila a ω_k = √(c²k² + m²)
omega_k = np.sqrt(c**2 * k**2 + m**2)

# ============== CONDICIÓN INICIAL ==============
# Envolvente gaussiana modulada por suma de modos cosenoidales
k0_arr = np.array(k0_sim).reshape(-1, 1)
A_arr = np.array(A_sim).reshape(-1, 1)
x_row = x.reshape(1, -1)

fase = np.sum(A_arr * (np.exp(1j * k0_arr * x_row) + np.exp(-1j * k0_arr * x_row)), axis=0)
psi_static = np.exp(-x**2 / (2 * sigma**2)) * fase
psi_moving = psi_static * np.exp(1j * k_boost * x)
psi_moving = psi_moving / np.sqrt(np.max(np.abs(psi_moving)**2))

phi0 = psi_moving.real
phi0 = phi0 / np.sqrt(np.max(np.abs(phi0)**2))

# π₀ = ∂φ/∂t|_{t=0} consistente con la relación de dispersión KG
phi_k_aux = np.fft.fft(psi_moving)
pi_k_aux = -1j * omega_k * phi_k_aux
pi0 = np.fft.ifft(pi_k_aux).real

# ============== PROPAGACIÓN EXACTA ==============
phi_k_0 = np.fft.fft(phi0)
pi_k_0 = np.fft.fft(pi0)

t_save = np.linspace(0.0, T_max, n_t)
Z_exact = np.zeros((n_t, N))

# Manejo del límite ω_k → 0 (solo ocurre si m=0 y k=0)
small = np.isclose(omega_k, 0.0)
omega_safe = np.where(small, 1.0, omega_k)

for i, t in enumerate(t_save):
    cos_wt = np.cos(omega_k * t)
    sin_wt = np.sin(omega_k * t)
    sin_wt_div_w = sin_wt / omega_safe
    sin_wt_div_w[small] = t  # lim_{ω→0} sin(ωt)/ω = t

    # Propagación exacta de cada modo de Fourier
    phi_k_t = phi_k_0 * cos_wt + pi_k_0 * sin_wt_div_w
    Z_exact[i, :] = np.fft.ifft(phi_k_t).real


In [ ]:
# 1. Parámetros del dominio (deben coincidir con el solver pseudo-espectral)
sigma = 1.0
L = 80 * sigma
X_min, X_max = -L/2, L/2
T_min, T_max = 0.0, L/2 - 4*sigma
N = 4096    # Puntos espaciales de la FFT (debe coincidir con el solver)
n_t = 200   # Snapshots almacenados

# 2. Generar la malla de prueba
# IMPORTANTE: endpoint=False en x para respetar la periodicidad de la FFT
x_test_phys = np.linspace(X_min, X_max, N, endpoint=False)
t_test_phys = np.linspace(T_min, T_max, n_t)
X_grid, T_grid = np.meshgrid(x_test_phys, t_test_phys)

# Z_exact debe haber sido calculado previamente con el solver pseudo-espectral
# (shape (n_t, N))

# 3. Predicción del modelo ABU-PINN
X_input = tf.cast(X_grid.reshape(-1, 1), tf.float64)
T_input = tf.cast(T_grid.reshape(-1, 1), tf.float64)

pred_out = model([X_input, T_input]).numpy()
Z_pred = pred_out.flatten().reshape(n_t, N)

# 4. Error Relativo L2
error_l2 = np.linalg.norm(Z_exact - Z_pred, ord='fro') / np.linalg.norm(Z_exact, ord='fro')
print(f"\n>>> Error Relativo L2 vs Referencia Pseudo-Espectral: {error_l2:.4e} <<<")

# 5. Graficar
extent = [X_min, X_max, T_min, T_max]
vmax = max(np.abs(Z_exact).max(), np.abs(Z_pred).max())

fig, axs = plt.subplots(1, 3, figsize=(20, 5))
cmap = "jet"

# Solución de referencia
im1 = axs[0].imshow(Z_exact, extent=extent, aspect='auto', origin='lower',
                    cmap=cmap)
axs[0].set_title("Solución Pseudo-Espectral (Referencia)")
axs[0].set_xlabel("x")
axs[0].set_ylabel("t")
plt.colorbar(im1, ax=axs[0])

# Predicción de la PINN
im2 = axs[1].imshow(Z_pred, extent=extent, aspect='auto', origin='lower',
                    cmap=cmap)
axs[1].set_title(f"PINN (Error L2: {error_l2:.2e})")
axs[1].set_xlabel("x")
plt.colorbar(im2, ax=axs[1])

# Error absoluto
im3 = axs[2].imshow(np.abs(Z_exact - Z_pred), extent=extent, aspect='auto',
                    origin='lower', cmap='inferno')
axs[2].set_title(r"Error Absoluto $|\phi_{exact} - \phi_{pred}|$")
axs[2].set_xlabel("x")
plt.colorbar(im3, ax=axs[2])

plt.tight_layout()
plt.show()

In [ ]:
# =====================  Verificación y ejecución con Qiskit (solo HQPINN)  =====================
if MODELO == "HQPINN":
    # 1) Los dos motores son el mismo circuito: forward y gradiente (parameter-shift vs autodiff)
    verificar_capa_contra_qiskit(model.vqc)
    verificar_gradiente_qiskit(model.vqc)

    # 2) φ en 2000 puntos de la malla de referencia con el VQC ejecutado por Qiskit Aer
    idx = np.random.default_rng(0).choice(X_input.shape[0], 2000, replace=False)
    xs, ts = tf.gather(X_input, idx), tf.gather(T_input, idx)
    ref = Z_exact.ravel()[idx]
    aer = model.vqc.ejecutor().estimador
    model.usar_motor("simulacion_exacta"); phi_sim = model([xs, ts]).numpy().ravel()
    model.usar_motor("qiskit_aer");        phi_qk = model([xs, ts]).numpy().ravel()          # Qiskit Aer, exacto
    model.usar_motor("simulacion_exacta")
    phi_sh = predecir_con_qiskit(model, xs, ts, shots=4096, estimador=aer).ravel()           # Qiskit Aer, 4096 shots
    l2 = lambda p: np.linalg.norm(p - ref) / np.linalg.norm(ref)
    print(f"max |φ_simulación − φ_Qiskit| = {np.max(np.abs(phi_sim - phi_qk)):.2e}")
    print(f"L2 en 2000 puntos: Qiskit Aer exacto = {l2(phi_qk):.4e} | Qiskit Aer 4096 shots = {l2(phi_sh):.4e} | "
          f"simulación exacta = {l2(phi_sim):.4e}")

    # 3) Diagrama del circuito
    print(model.vqc.qc.draw(output="text", fold=120))

    # 4) (Opcional) hardware de IBM — requiere cuenta y token:
    # from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2
    # from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    # service = QiskitRuntimeService(); backend = service.least_busy(operational=True, simulator=False)
    # isa = generate_preset_pass_manager(backend=backend, optimization_level=1).run(model.vqc.qc)
    # obs = [o.apply_layout(isa.layout) for o in observables_z(model.vqc.n_qubits)]
    # ... evaluar_con_qiskit(isa, ...) con estimador=EstimatorV2(mode=backend) y esos observables


In [ ]:
# Guardar el modelo completo de Keras
modelo_path = '/content/ABUN_PINN/PINN_model.keras'
model.save(modelo_path)
print(f"Modelo completo guardado exitosamente en: {modelo_path}")

In [ ]:
import pandas as pd
import numpy as np

# Calcular el error absoluto medio (Mean Absolute Error)
error_abs = np.mean(np.abs(Z_exact - Z_pred))

# Número de parámetros del modelo
num_params = model.count_params()

# Función de costo promedio de las última capa
costo = df_loss['total'].iloc[-1]

# Crear la tabla de errores ampliada
df_errores = pd.DataFrame(
    {
        "Error Relativo L2": [f"{error_l2:.2f}"],
        "Error Absoluto": [f"{error_abs:.4f}"],
        "Número de parámetros": [f"{num_params}"],
        "Costo final": [f"{costo:.2e}"]
    },
    index=["PINN"]
)

# Mostrar la tabla
df_errores

In [ ]:
# Guardar la tabla de errores en un archivo CSV
csv_path = '/content/ABUN_PINN/errores_PINN.csv'
df_errores.to_csv(csv_path, index=True)
print(f"Tabla de errores guardada exitosamente en: {csv_path}")